# MAS: Multi-Agent System for Airport Anomaly Detection

This notebook implements an agentic workflow designed to detect and analyze security anomalies in airport passenger traffic. By utilizing a **Multi-Agent System (MAS)** built with `LangGraph` and local LLMs via `Ollama`, we separate data processing from high-level analytical reasoning.

### Key Objectives:
* **Contextual Data Ingestion**: Automatically filtering and cleaning passenger and alarm datasets based on user-defined perimeters.
* **Time-Series Math Integration**: Using classical statistical methods (Rolling Averages, Z-Scores) to anchor AI reasoning in mathematical reality.
* **Automated Situational Reporting**: Generating human-readable security reports that explain anomalies using both historical trends and demographic context.

In [1]:
import pandas as pd
import numpy as np
import operator
import io
import json  
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

## 2. Defining the Shared Memory (State)
In an agentic system, the **State** acts as the shared memory. It allows agents to pass structured data (like JSON strings) and natural language messages to one another.

* `perimeter`: The target airport IATA code (e.g., FCO, MXP, LIN).
* `passenger_json` / `alarm_json`: Cleaned datasets processed by the Data Agent.
* `baseline_stats`: The "Golden Standard" of math produced by the Baseline Agent.

In [2]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    perimeter: str
    passenger_json: str  
    alarm_json: str      
    baseline_stats: str

## 3. Data Extraction & Normalization

### Italian Date Normalization (`clean_italian_dates`)
One of the primary challenges in this dataset is the presence of non-standard Italian date formats (e.g., "GEN" for January). 
The `clean_italian_dates` function performs the following:
1.  **Translation**: Maps Italian month abbreviations (GEN, MAG, DIC) to English equivalents.
2.  **Robust Parsing**: Uses Pandas' `mixed` format parsing with `dayfirst=True` to ensure consistency across different CSV formats.


In [3]:
import re

def clean_italian_dates(series):
    # mapping italian abbreviations to english 
    month_map = {
        'GEN': 'JAN', 'FEB': 'FEB', 'MAR': 'MAR', 'APR': 'APR',
        'MAG': 'MAY', 'GIU': 'JUN', 'LUG': 'JUL', 'AGO': 'AUG',
        'SET': 'SEP', 'OTT': 'OCT', 'NOV': 'NOV', 'DIC': 'DEC'
    }
    
    def replace_month(date_str):
        if pd.isna(date_str): return date_str
        date_str = str(date_str).upper()
        for it, en in month_map.items():
            if it in date_str:
                return date_str.replace(it, en)
        return date_str

    # apply translation and then parse with mixed format
    translated = series.apply(replace_month)
    return pd.to_datetime(translated, format='mixed', dayfirst=True, errors='coerce')

### Security Context Fetching (`fetch_security_context`)
This function acts as the "Retriever" for our system. It filters the raw CSVs based on the user's input perimeter, cleans numeric columns (handling missing or malformed data), and prepares the dataframes for analysis.

In [4]:
def fetch_security_context(perimeter: str):
    p_up = perimeter.upper().strip()
    
    # passengers data
    df_p = pd.read_csv('TIPOLOGIA_VIAGGIATORE.csv', sep=None, engine='python')
    mask_p = (df_p['AREOPORTO_ARRIVO'].str.upper() == p_up) | \
             (df_p['CITTA_ARR'].str.upper().str.contains(p_up, na=False))
    p_df = df_p[mask_p].copy()
    
    # numeric and dates cleaning
    for col in ['ENTRATI', 'INVESTIGATI', 'ALLARMATI']:
        p_df[col] = pd.to_numeric(p_df[col], errors='coerce').fillna(0).astype(int)
    
    p_df['timestamp'] = clean_italian_dates(p_df['DATA_PARTENZA'])
    p_df = p_df.dropna(subset=['timestamp']) # Remove unparseable dates
    p_df['join_date'] = p_df['timestamp'].dt.date

    # alarms data
    df_a = pd.read_csv('ALLARMI.csv', sep=None, engine='python')
    mask_a = (df_a['AREOPORTO_ARRIVO'].str.upper() == p_up) | \
             (df_a['CITTA_ARR'].str.upper().str.contains(p_up, na=False))
    a_df = df_a[mask_a].copy()
    
    if 'TOT' in a_df.columns:
        a_df['TOT'] = pd.to_numeric(a_df['TOT'], errors='coerce').fillna(0).astype(int)

    a_df['timestamp'] = clean_italian_dates(a_df['DATA_PARTENZA'])
    a_df = a_df.dropna(subset=['timestamp'])
    a_df['join_date'] = a_df['timestamp'].dt.date
    
    return p_df, a_df

## 4. Analytical Tools: The Anti-Hallucination Engine

### Statistical Logic (`calculate_security_stats`)
LLMs often struggle with precise mathematical calculations over large datasets, sometimes hallucinating impossible percentages. To solve this, we implement a **Classical Math Tool**. 

This function calculates:
* **7-Observation Rolling Averages**: Handles irregular data gaps by looking at the last 7 actual data points rather than calendar days.
* **Deviation Ratios**: Compares current activity to the recent 7-point average to identify sudden spikes.
* **Monthly Z-Scores**: Standardizes traffic volume against the historical monthly mean to account for seasonal variations.
* **National Demographics**: Summarizes the top traveler nationalities involved in the latest observations.

In [5]:
def calculate_security_stats(passenger_data: str, alarm_data: str):
    p_df = pd.read_json(io.StringIO(passenger_data), orient='records')
    a_df = pd.read_json(io.StringIO(alarm_data), orient='records')
    
    p_df['timestamp'] = pd.to_datetime(p_df['timestamp'])
    a_df['timestamp'] = pd.to_datetime(a_df['timestamp'])
    
    # 1. aggregate observations
    # we include investigations here to match the classical model's metrics
    daily_pass = p_df.groupby(p_df['timestamp'].dt.date).agg(
        {'ENTRATI': 'sum', 'INVESTIGATI': 'sum'}
    ).reset_index()
    daily_pass.columns = ['date', 'entries', 'investigations']
    
    real_alarms = a_df[a_df['OCCORRENZE'].str.contains('Allarmi', na=False, case=False)]
    daily_alarms = real_alarms.groupby(real_alarms['timestamp'].dt.date)['TOT'].sum().reset_index()
    daily_alarms.columns = ['date', 'alarms']
    
    # merge into a single observation timeline
    df = pd.merge(daily_pass, daily_alarms, on='date', how='left').fillna(0)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    # 2. calculate rates
    df['alarm_rate'] = df['alarms'] / df['entries'].replace(0, 1)
    df['inv_rate'] = df['investigations'] / df['entries'].replace(0, 1)
    
    # 3. 7-observation rolling averages
    # Using min_periods=1 handles the 25% of routes with sparse data safely
    df['entries_7_mean'] = df['entries'].rolling(window=7, min_periods=1).mean()
    df['alarm_rate_7_mean'] = df['alarm_rate'].rolling(window=7, min_periods=1).mean()
    
    # 4. deviation ratios
    df['entries_dev_ratio'] = df['entries'] / df['entries_7_mean'].replace(0, 1)
    df['alarm_dev_ratio'] = df['alarm_rate'] / df['alarm_rate_7_mean'].replace(0, 1)
    
    # 5. monthly seasonal baseline and z-score
    df['month'] = df['date'].dt.month
    monthly_stats = df.groupby('month')['entries'].agg(['mean', 'std']).reset_index()
    monthly_stats.rename(columns={'mean': 'monthly_mean', 'std': 'monthly_std'}, inplace=True)
    
    df = pd.merge(df, monthly_stats, on='month', how='left')
    
    df['monthly_std'] = df['monthly_std'].fillna(1).replace(0, 1)
    df['entries_z_score'] = (df['entries'] - df['monthly_mean']) / df['monthly_std']
    
    # 6. extract 'current' state for the agent
    latest = df.iloc[-1]
    
    nat_counts = p_df['NAZIONALITA'].value_counts(normalize=True).head(3)
    nat_summary = [f"{k} ({round(v*100, 1)}%)" for k, v in nat_counts.items()]

    return {
        "latest_observation_date": str(latest['date'].date()),
        "current_entries": int(latest['entries']),
        "monthly_z_score": round(float(latest['entries_z_score']), 2),
        "current_alarm_rate": f"{round(float(latest['alarm_rate']) * 100, 2)}%",
        "alarm_deviation_ratio": round(float(latest['alarm_dev_ratio']), 2),
        "7_obs_historical_alarm_rate": f"{round(float(latest['alarm_rate_7_mean']) * 100, 2)}%",
        "top_nationalities": ", ".join(nat_summary)
    }

## 5. AI Agent Nodes

### Data Agent (`data_agent_node`)
This agent is responsible for intent recognition. It uses the LLM to extract a clean IATA code from user input and triggers the data ingestion pipeline. It serves as the entry point for the graph.


In [6]:
def data_agent_node(state: AgentState):
    llm = ChatOllama(model="llama3.2:3b", temperature=0)
    prompt = f"Extract ONLY the 3-letter IATA code from: '{state['messages'][-1].content}'. No words, just the code (e.g. FCO)."
    perimeter = llm.invoke([HumanMessage(content=prompt)]).content.strip().upper()
    
    # regex safety to ensure we have a clean code
    perimeter = re.findall(r'[A-Z]{3}', perimeter)[0] if re.findall(r'[A-Z]{3}', perimeter) else perimeter
    
    p_df, a_df = fetch_security_context(perimeter)
    print(f"--- 🕵️ DATA AGENT: {perimeter} ({len(p_df)} records) ---")

    return {
        "perimeter": perimeter,
        "passenger_json": p_df.to_json(orient='records', date_format='iso'),
        "alarm_json": a_df.to_json(orient='records', date_format='iso')
    }

from langchain_core.tools import tool



### Baseline Agent (`baseline_agent_node`)
Acting as a **Senior Border Security Analyst**, this agent:
1.  Calls the math tools to get precise metrics.
2.  Synthesizes the math with demographic context.
3.  Writes a neutral, objective situational report that avoids bias while highlighting significant statistical deviations.

In [7]:
@tool
def security_math_tool(passenger_json: str, alarm_json: str):
    """Use this tool to calculate precise security statistics and anomaly thresholds."""
    return calculate_security_stats(passenger_json, alarm_json)

def baseline_agent_node(state: AgentState):
    print("--- 📊 BASELINE AGENT: Evaluating Time-Series Context ---")
    
    tool_result = calculate_security_stats(state["passenger_json"], state["alarm_json"])
    
    llm = ChatOllama(model="llama3.2:3b", temperature=0)
    
    report_prompt = f"""
    ROLE: Senior Border Security Analyst.
    AIRPORT: {state['perimeter']} (Milan Linate).
    LATEST DATA POINT: {tool_result}
    
    INSTRUCTIONS ON METRICS:
    - 'monthly_z_score': A value > 2 means traffic is unusually high for this month.
    - 'alarm_deviation_ratio': A value > 3 means the alarm rate is 3x higher than the recent 7-flight average.
    
    TASK: Write a situational report for the latest observation date.
    1. Report the current alarm rate vs the historical 7-observation rate.
    2. Analyze the context using the Z-score and Deviation Ratio.
    3. Maintain an objective, non-biased tone and mention top nationalities. Do not hallucinate city names.

    IMPORTANT: Avoid making speculative assumptions about traveler risk based solely on nationality.
    """
    
    response = llm.invoke([HumanMessage(content=report_prompt)])
    return {
        "baseline_stats": json.dumps(tool_result),
        "messages": [response]
    }

## 6. Data Integrity Check
Before running the graph, we perform a quick audit of the source files to ensure we have valid coverage for the major Italian perimeters (NAP, FCO, LIN, MXP, etc.). This ensures the system has a valid target before the agents begin processing.

In [8]:

df_audit = pd.read_csv('TIPOLOGIA_VIAGGIATORE.csv', sep=None, engine='python')

print("📍 Unique Airport Codes in CSV:")
print(df_audit['AREOPORTO_ARRIVO'].unique()[:10]) # Show first 10

print("\n🏙️ Unique Cities in CSV:")
print(df_audit['CITTA_ARR'].unique()[:10])

📍 Unique Airport Codes in CSV:
['NAP' 'FCO' 'TSF' 'BGY' 'PSA' 'BLQ' 'MXP' 'LIN' 'VRN' 'BRI']

🏙️ Unique Cities in CSV:
['Napoli' 'Roma' 'Treviso' 'Bergamo' 'Pisa' 'Bologna' 'Milano' 'Verona'
 'Bari' 'Catania']


## 7. The Workflow Graph
Using `StateGraph`, we define the "brain" of the operation. The workflow is a Directed Acyclic Graph (DAG) that ensures the **Data Agent** completes its task before passing the context to the **Baseline Agent**. We also integrate `MemorySaver` to allow for threaded conversations.

In [9]:
builder = StateGraph(AgentState)
builder.add_node("data_agent", data_agent_node)
builder.add_node("baseline_agent", baseline_agent_node)
builder.set_entry_point("data_agent")
builder.add_edge("data_agent", "baseline_agent")
builder.add_edge("baseline_agent", END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)



## 8. Executing the Security Analysis
This is the final execution block. The user identifies a perimeter, and the MAS generates a finalized report. 
**Note**: If you see a 100% alarm rate for low-volume perimeters (like LIN), the agent correctly identifies this as a "low entries" incident rather than a systemic failure, demonstrating the power of combined math and reasoning.

In [10]:
user_q = input("Identify target perimeter (e.g. Rome, Bologna, MXP): ")
if user_q:
    config = {"configurable": {"thread_id": "v1"}}
    for event in app.stream({"messages": [HumanMessage(content=user_q)]}, config, stream_mode="values"):
        final_state = event
    
    print(f"\n🛡️ OFFICIAL REPORT:\n{final_state['messages'][-1].content}")

Identify target perimeter (e.g. Rome, Bologna, MXP):  Rome


--- 🕵️ DATA AGENT: FCO (882 records) ---
--- 📊 BASELINE AGENT: Evaluating Time-Series Context ---

🛡️ OFFICIAL REPORT:
Situational Report for FCO (Milan Linate) Airport - February 29, 2024

Introduction:
This report provides an analysis of the latest observation date at FCO airport, highlighting key metrics and trends in border security.

Current Alarm Rate vs Historical Average:
The current alarm rate stands at 16.48%, which is significantly higher than the recent 7-observation average of 13.44%. This indicates a notable increase in alarm events compared to the previous week.

Context Analysis:

1. **Monthly Z-Score**: The monthly z-score value of -0.71 suggests that, relative to historical norms, the current month's traffic is slightly below average. However, it is essential to note that this metric alone does not provide a comprehensive understanding of the situation.
2. **Alarm Deviation Ratio**: The alarm deviation ratio of 1.23 indicates that the current alarm rate is approximate